# bot

> Discord passthrough to a headless coding agent CLI: subprocess runner,
> reply shaping, env config, and the thin discord.py glue.

In [ ]:
#| default_exp bot

## `run_agent`

Run one agent turn as a subprocess: stdin ← message, capture stdout and
stderr **separately** (claude emits warnings on stderr even on success).
Timeout: SIGKILL the whole process group, recover partial output via a
shielded `communicate()` (an unshielded `wait_for` drops the buffer).
Never raises — nonzero exit is data, not an exception, and is **never
retried** (a retry would replay side effects like PR creation).

In [ ]:
#| export
import asyncio, os, signal

async def run_agent(text:str,            # message for the agent, fed to stdin
                    cmd:list,            # agent command line, already shlex-split
                    cwd:str,             # working directory for the agent
                    timeout:float=600.0, # wall-clock seconds before SIGKILL
                    )->tuple:            # (stdout, stderr, returncode)
    "Run one agent turn; never raises on nonzero exit or timeout."
    proc = await asyncio.create_subprocess_exec(
        *cmd, cwd=cwd, start_new_session=True,
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
    comm = asyncio.ensure_future(proc.communicate(text.encode()))
    try:
        out, err = await asyncio.wait_for(asyncio.shield(comm), timeout)
    except asyncio.TimeoutError:
        try: os.killpg(proc.pid, signal.SIGKILL)   # whole session: agent + children
        except ProcessLookupError: pass
        try:
            out, err = await asyncio.wait_for(asyncio.shield(comm), 1)
        except asyncio.TimeoutError:
            comm.cancel()
            return "… timed out (output withheld by a background child)\n", "", -9   # -9: killed by SIGKILL
        rc = proc.returncode if proc.returncode is not None else -9
        return out.decode(errors="replace") + "\n… timed out\n", err.decode(errors="replace"), rc
    return out.decode(errors="replace"), err.decode(errors="replace"), proc.returncode

In [ ]:
out, err, rc = await run_agent("hello", ["bash", "-c", "cat; echo warn >&2"], ".", 5)
assert out == "hello", out                    # stdin delivered, stdout captured
assert "warn" in err and "warn" not in out    # stderr kept separate
assert rc == 0

out, err, rc = await run_agent("", ["bash", "-c", "echo partial; echo boom >&2; exit 3"], ".", 5)
assert rc == 3                                # nonzero exit reported, not raised
assert "partial" in out and "boom" in err

out, err, rc = await run_agent("", ["bash", "-c", "echo start; sleep 5"], ".", 0.5)
assert "start" in out, out                    # partial output recovered
assert "timed out" in out and rc != 0

out, err, rc = await run_agent("", ["bash", "-c", "echo hi; sleep 60 &"], ".", 1)
assert "hi" in out and "timed out" in out     # killpg reaps the background child